# Phase 0 — Environment and dataset inventory

**Objective:** Confirm which project files, Python environment, dataset revisions, and hardware are available before preprocessing or training.

This notebook records observations only. Counts calculated here are not final training-set counts. The train/validation/test split has not yet been created.

In [12]:
from pathlib import Path
import sys
import platform
from datetime import datetime

# Search upward from the notebook's current working directory for the BTP root.
# This works when the notebook is in BTP/ or BTP/notebooks/.
current = Path.cwd().resolve()
project = next(
    (
        folder for folder in (current, *current.parents)
        if (folder / "manifests").is_dir()
        and (folder / "raw" / "A_narration").is_dir()
    ),
    None,
)

if project is None:
    raise FileNotFoundError(
        "BTP was not found above the notebook's working directory. "
        f"Current directory: {current}"
    )

print("Recorded at:", datetime.now().astimezone().isoformat(timespec="seconds"))
print("Project directory:", project)
print("Notebook working directory:", current)
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Operating system:", platform.platform())


# Check the environment Python reports, without following the executable symlink.
if Path(sys.prefix).absolute() == (project / ".venv").absolute():
    print("Kernel check: using the BTP .venv")
else:
    print("Kernel check: review your kernel selection")
    print("Active environment:", sys.prefix)

Recorded at: 2026-10-31T19:03:45+05:30
Project directory: /Users/kanchanpotdar/Downloads/BTP
Notebook working directory: /Users/kanchanpotdar/Downloads/BTP/notebooks
Python executable: /Users/kanchanpotdar/Downloads/BTP/.venv/bin/python
Python version: 3.11.1
Operating system: macOS-26.5.1-arm64-arm-64bit
Kernel check: using the BTP .venv


## 1. Confirm the project inputs

List the files needed for the Narration baseline. A missing optional review file is a question to investigate; a missing raw dataset or master manifest blocks later phases.

In [3]:
# These are the current project inputs, not files this notebook will create.
inputs = {
    "Narration raw data": project / "raw" / "A_narration",
    "Starting master": project / "manifests" / "master_A_v1_1.parquet",
    "Original master": project / "manifests" / "master_A.parquet",
    "Audio decode facts": project / "manifests" / "audio_decode_facts_A.parquet",
    "Speaker proposals": project / "review" / "speaker_groups.csv",
    "Speaker conflicts": project / "review" / "speaker_conflicts.csv",
    "IPA inventory": project / "review" / "ipa_inventory.csv",
    "Previous notebook findings": project / "docs" / "NOTEBOOK_FINDINGS.md",
    "Research decisions": project / "DECISIONS.md",
    "Sprint plan": project / "SPRINT_PLAN.md",
}

for description, path in inputs.items():
    state = "FOUND" if path.exists() else "MISSING"
    print(f"{state:7} {description:27} {path.relative_to(project)}")

# Phase 0 should identify missing inputs, not silently recreate them.
critical = ("Narration raw data", "Starting master", "Audio decode facts")
missing_critical = [name for name in critical if not inputs[name].exists()]
print("\nCritical missing inputs:", missing_critical or "none")

FOUND   Narration raw data          raw/A_narration
FOUND   Starting master             manifests/master_A_v1_1.parquet
FOUND   Original master             manifests/master_A.parquet
FOUND   Audio decode facts          manifests/audio_decode_facts_A.parquet
FOUND   Speaker proposals           review/speaker_groups.csv
FOUND   Speaker conflicts           review/speaker_conflicts.csv
FOUND   IPA inventory               review/ipa_inventory.csv
FOUND   Previous notebook findings  docs/NOTEBOOK_FINDINGS.md
FOUND   Research decisions          DECISIONS.md
FOUND   Sprint plan                 SPRINT_PLAN.md

Critical missing inputs: none


In [4]:
# Inspect file presence and sizes without decoding thousands of audio clips.
narration_files = sorted((project / "raw" / "A_narration").rglob("*.parquet"))
youtube_files = sorted((project / "raw" / "B_youtube").rglob("*.parquet"))

def gib_total(paths):
    return sum(path.stat().st_size for path in paths) / (1024 ** 3)

print(f"Narration Parquet files: {len(narration_files)}")
print(f"Narration disk size:     {gib_total(narration_files):.2f} GiB")
print(f"YouTube Parquet files:   {len(youtube_files)}")
print(f"YouTube disk size:       {gib_total(youtube_files):.2f} GiB")
print("\nFirst Narration paths:")
for path in narration_files[:5]:
    print(" ", path.relative_to(project))

# The YouTube count is inventory only; Dataset A remains the first experiment.

Narration Parquet files: 59
Narration disk size:     1.57 GiB
YouTube Parquet files:   6
YouTube disk size:       1.59 GiB

First Narration paths:
  raw/A_narration/data/test/data-083f238c.parquet
  raw/A_narration/data/test/data-2cf54942.parquet
  raw/A_narration/data/test/data-756603b7.parquet
  raw/A_narration/data/test/data-7ef0d7c8.parquet
  raw/A_narration/data/test/data-aefffea7.parquet


## 2. Inspect the master without changing it

The starting master is a derived metadata table. We will inspect its shape, column names, and a few selected fields. Avoid displaying full transcripts or embedded audio in notebook output.

In [5]:
try:
    import pyarrow.parquet as pq
except ImportError as exc:
    raise ImportError(
        "This kernel cannot import pyarrow. Check that Python (BTP) is selected "
        "before installing or changing packages."
    ) from exc

master_path = project / "manifests" / "master_A_v1_1.parquet"
parquet_file = pq.ParquetFile(master_path)

# Metadata inspection does not load the whole dataset or decode audio.
columns = parquet_file.schema_arrow.names
print("Master:", master_path.relative_to(project))
print("Rows:", parquet_file.metadata.num_rows)
print("Columns:", len(columns))
print("Row groups:", parquet_file.num_row_groups)
print("\nColumn names:")
for name in columns:
    print(" ", name)

Master: manifests/master_A_v1_1.parquet
Rows: 6718
Columns: 54
Row groups: 1

Column names:
  sample_id
  source_dataset
  original_audio_id
  audio_path
  original_split
  start_time
  end_time
  duration_seconds
  duration_basis
  transcript_raw
  transcript_missing
  transcript_normalized
  speaker_id
  speaker_id_raw
  speaker_status
  is_pair_recording
  train_only
  recording_session_id
  sentence_id
  boundary_id
  filename
  domain
  elicitation_method
  q_id
  gender
  age_group
  education_level
  place_of_recording
  type_of_place
  other_languages
  speaker2_gender
  speaker2_age_group
  quality_status
  exclusion_reason
  transcript_review_status
  language_status
  content_type
  duplicate_group_id
  final_split
  speaker_group_id
  group_train_only
  recording_month
  education_medium_after_12
  education_medium_upto_12
  life_speaker_id
  speaker2_place_of_recording
  speaker2_type_of_place
  audio_subtype
  audio_sr
  audio_channels
  duration_decoded
  length_matches_

In [6]:
# Different historical manifests use different column names. Inspect the
# actual schema before choosing a transcript or duration column.
terms = ("transcript", "text", "duration", "speaker", "session", "split", "audio", "id")
matches = [name for name in columns if any(term in name.lower() for term in terms)]

print("Columns likely relevant to later phases:")
for name in matches:
    field = parquet_file.schema_arrow.field(name)
    print(f"  {name}: {field.type}")

print(
    "\nUse the printed names to identify the raw transcript, decoded duration, "
    "audio status, sample ID, and speaker fields. Do not assume a similarly "
    "named provisional normalization column is the approved target."
)

Columns likely relevant to later phases:
  sample_id: string
  original_audio_id: string
  audio_path: string
  original_split: string
  duration_seconds: double
  duration_basis: string
  transcript_raw: string
  transcript_missing: bool
  transcript_normalized: null
  speaker_id: string
  speaker_id_raw: string
  speaker_status: string
  recording_session_id: string
  sentence_id: string
  boundary_id: string
  q_id: string
  speaker2_gender: string
  speaker2_age_group: string
  transcript_review_status: string
  duplicate_group_id: null
  final_split: null
  speaker_group_id: string
  life_speaker_id: string
  speaker2_place_of_recording: string
  speaker2_type_of_place: string
  audio_subtype: string
  audio_sr: int64
  audio_channels: int64
  duration_decoded: double
  audio_ok: bool
  audio_issue: string

Use the printed names to identify the raw transcript, decoded duration, audio status, sample ID, and speaker fields. Do not assume a similarly named provisional normalization c

In [7]:
# Parquet footers contain row counts. This avoids loading the audio payloads.
shard_rows = []
for path in narration_files:
    metadata = pq.read_metadata(path)
    shard_rows.append((path.relative_to(project).as_posix(), metadata.num_rows))

print("Raw Narration shard count:", len(shard_rows))
print("Raw Narration row total:", sum(count for _, count in shard_rows))
print("Starting master row total:", parquet_file.metadata.num_rows)

if sum(count for _, count in shard_rows) != parquet_file.metadata.num_rows:
    print("REVIEW: Raw shard and master row totals differ.")
else:
    print("Row-count check: totals agree.")

print("\nRows by raw shard:")
for name, count in shard_rows:
    print(f"{count:5d}  {name}")

Raw Narration shard count: 59
Raw Narration row total: 6718
Starting master row total: 6718
Row-count check: totals agree.

Rows by raw shard:
   50  raw/A_narration/data/test/data-083f238c.parquet
  426  raw/A_narration/data/test/data-2cf54942.parquet
   50  raw/A_narration/data/test/data-756603b7.parquet
   51  raw/A_narration/data/test/data-7ef0d7c8.parquet
   47  raw/A_narration/data/test/data-aefffea7.parquet
   60  raw/A_narration/data/test/data-babc44cf.parquet
  117  raw/A_narration/data/train/data-0d68d838.parquet
   78  raw/A_narration/data/train/data-17ae335f.parquet
   50  raw/A_narration/data/train/data-1867a0cb.parquet
   50  raw/A_narration/data/train/data-1aef74ba.parquet
   50  raw/A_narration/data/train/data-1d7e6898.parquet
  100  raw/A_narration/data/train/data-1eab3e46.parquet
  823  raw/A_narration/data/train/data-1ffa879a.parquet
   50  raw/A_narration/data/train/data-21de5da3.parquet
   47  raw/A_narration/data/train/data-23be8ad5.parquet
   72  raw/A_narration/

## 3. Record software and hardware

Local Phase 0 can be completed without a GPU. GPU availability, memory, and quota must be measured again inside the actual Kaggle or Colab session used for training.

In [8]:
from importlib.metadata import version, PackageNotFoundError

# Record the environment actually running this notebook.
packages = (
    "numpy", "pandas", "pyarrow", "soundfile",
    "torch", "transformers", "datasets", "evaluate",
    "accelerate", "jiwer", "ipykernel",
)

for package in packages:
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = "not installed"
    print(f"{package:15} {installed}")

# A missing training package is a setup item for the pilot, not a data error.

numpy           2.3.5
pandas          2.2.3
pyarrow         26.0.0
soundfile       0.13.1
torch           not installed
transformers    not installed
datasets        not installed
evaluate        not installed
accelerate      not installed
jiwer           not installed
ipykernel       7.4.0


In [10]:
import shutil

disk = shutil.disk_usage(project)
print(f"Free disk space at BTP: {disk.free / (1024 ** 3):.1f} GiB")

try:
    import torch
except ImportError:
    print("PyTorch is not installed in this kernel; GPU check unavailable.")
else:
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(index)
            print(
                f"CUDA GPU {index}: {properties.name}, "
                f"{properties.total_memory / (1024 ** 3):.1f} GiB VRAM"
            )

    # Apple Silicon may support MPS for local experiments.
    mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    print("Apple MPS available:", mps_available)

Free disk space at BTP: 141.7 GiB
PyTorch is not installed in this kernel; GPU check unavailable.


## Phase 0 conclusion

**Status:** Local environment and dataset inventory complete. Kaggle hardware inventory remains open.

- **Recorded:** 31 October 2026, in `/Users/kanchanpotdar/Downloads/BTP`.
- **Notebook kernel:** BTP `.venv`, Python 3.11.1. The old “outside the BTP .venv” message came from an incorrect check; the corrected check confirms the project environment.
- **Data available:** 59 Narration Parquet shards containing 6,718 rows. Their total matches the 6,718 rows in `manifests/master_A_v1_1.parquet`, which has 54 columns. All critical local inputs were found.
- **Starting eligibility facts:** 5,651 rows are flagged as having transcripts; 1,067 are flagged as missing transcripts. Audio is marked OK for 6,714 rows; three clips have zero decoded duration. There are 708 clips longer than 30 seconds. A first-pass check gives 5,086 labelled, audio-OK clips between 0.5 and 30 seconds. **This is a candidate pool, not a final training split.**
- **Local environment:** NumPy 2.3.5, pandas 2.2.3, PyArrow 26.0.0, SoundFile 0.13.1, and ipykernel 7.4.0 are available. PyTorch, Transformers, Datasets, Evaluate, Accelerate, and jiwer are not installed locally.
- **Local storage:** 141.7 GiB free when measured.
- **GPU:** Not measured locally because PyTorch is absent. Kaggle GPU type, VRAM, available quota, and package versions remain to be checked in the actual Kaggle session.

**Columns to carry into Phase 1:** `sample_id`, `audio_path`, `original_audio_id`, `transcript_raw`, `transcript_missing`, `audio_ok`, `audio_issue`, `duration_decoded`, `duration_seconds`, `length_matches_stated`, `audio_sr`, and `audio_channels`. Retain speaker, session, and source columns for the later split audit.

**Open questions:** Confirm each eligibility rule and its row/hour count in Phase 1; review the known duration discrepancy; verify proposed speaker groups before freezing splits. `transcript_normalized`, `duplicate_group_id`, and `final_split` are currently empty and do not represent completed work.

**Decision:** Ready to start local Phase 1 data-quality and eligibility checks. Check Kaggle hardware before the model smoke test or training pilot.